In [2]:
import os
import pandas as pd
import numpy as np
import google.generativeai as genai

from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from sklearn.metrics.pairwise import cosine_similarity



e:\Graduation Project\Chatbot\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\mazen\AppData\Local\Temp\ipykernel_16792\2014627833.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")


In [5]:
if not GOOGLE_API_KEY:
    raise ValueError("API Key is not found! Make sure you have a .env file.")

In [6]:
genai.configure(api_key=GOOGLE_API_KEY)

print("Gemini API is configured successfully!")

Gemini API is configured successfully!


In [7]:
PDF_URL = "https://www.goabroadedu.in/wp-content/uploads/2017/07/Oxford-Handbook-of-Clinical-Haematology-4e.pdf"




In [7]:
for model in genai.list_models():
    if 'generateContent' in model.supported_generation_methods:
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gem

In [8]:

def get_response(prompt: str, model_name: str = "gemini-2.5-flash") -> str:
    model = genai.GenerativeModel(model_name)
    response = model.generate_content(prompt)
    return response.text.strip()

In [9]:
get_response("What is the capital of France?")

'The capital of France is **Paris**.'

In [10]:
documents = PyPDFLoader(PDF_URL).load_and_split()


PdfReadError("Invalid Elementary Object starting with b'P' @4300: b'8 0 obj<</Universal PDF(The process that creates this PDF constitutes a trade se'")


In [15]:
print(f"Loaded {len(documents)} pages.")

Loaded 871 pages.


In [16]:
pip install pymupdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from langchain_community.document_loaders import PyMuPDFLoader

In [11]:
documents = PyMuPDFLoader(PDF_URL).load_and_split()

In [12]:
print(f"Loaded {len(documents)} pages.")

Loaded 871 pages.


In [13]:
def splitter( chunk_size: int = 1024, chunk_overlap: int = 256):
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )
        return splitter.split_documents(documents)



In [14]:
chunks = splitter()
print(f"Split into {len(chunks)} chunks.")

Split into 2123 chunks.


In [15]:
import torch
torch.cuda.is_available()

True

In [24]:
pip uninstall torch torchvision torchaudio -y

Found existing installation: torch 2.11.0
Uninstalling torch-2.11.0:
  Successfully uninstalled torch-2.11.0
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.


In [25]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp312-cp312-win_amd64.whl (2449.3 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp312-cp312-win_amd64.whl (6.1 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchaudio-2.5.1%2Bcu121-cp312-cp312-win_amd64.whl (4.1 MB)
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
Using cached sympy-1.13.1-py3-none-any.whl (6.2 MB)
   ---------------------------------------- 0.0/7.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.0 MB 320.0 kB/s eta 0:00:22
   ---------------------------------------- 0.0/7.0 MB 320.0 kB/s eta 0:00:22
   ---------------------------------------- 0.0/7.0 MB 325.1 kB/s eta 0:00:22
   ---------------------------------------- 0.1/7.0 MB 409.6 kB/s eta 0:00:18
    ----------------------


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import torch
torch.cuda.is_available()

True

In [16]:
def build_or_load_vector_store(texts=None, faiss_dir="./faiss_index", embedding_model_name="intfloat/multilingual-e5-large-instruct"):
    
    print(" Loading embedding model …")  
    embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name,model_kwargs={'device': 'cuda'})
    print(" Embedding model ready.")

    if os.path.exists(faiss_dir):
        print(f" Loading existing FAISS index from '{faiss_dir}' …")
        faiss_index = FAISS.load_local(
            faiss_dir,
            embeddings,
            allow_dangerous_deserialization=True,
        )
        print(" FAISS index loaded.")
        
    else:
        if texts is None:
            raise ValueError("Texts must be provided to build the FAISS index from scratch.")
            
        print(" Building FAISS index from scratch …")
        print(f"    → {len(texts)} chunks created, embedding now …")
        
        faiss_index = FAISS.from_documents(texts, embeddings)
        faiss_index.save_local(faiss_dir)
        print(f" FAISS index saved to '{faiss_dir}'.")

    return faiss_index

In [17]:
build_or_load_vector_store(chunks)

 Loading embedding model …


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1497.48it/s]


 Embedding model ready.
 Building FAISS index from scratch …
    → 2123 chunks created, embedding now …
 FAISS index saved to './faiss_index'.


In [18]:
def retrieval_with_score(
    question: str,
    faiss_index, 
    k: int = 4,
    score_threshold: float = 1.0,
    verbose: bool = True) -> str:

    prefixed_question = (
        "Instruct: Given a question, retrieve relevant passages\nQuery: "
        + question
    )
    
    matched_docs = faiss_index.similarity_search_with_score(
        prefixed_question,
        k=k,
        score_threshold=score_threshold,
    )

    context = ""
    log_chunks = ""
    for i, (doc, chunk_score) in enumerate(matched_docs):
        context += doc.page_content + "\n"
        preview = (
            doc.page_content[:300] + "…"
            if len(doc.page_content) > 300
            else doc.page_content
        )
        log_chunks += (
            f"\n[Chunk {i+1}] "
            f"Page: {doc.metadata.get('page', 'N/A')} | "
            f"L2 Score: {chunk_score:.4f}\n{preview}\n"
        )

    if verbose:
        print(f"\n{'=' * 60}")
        print("  RETRIEVED CHUNKS")
        print('=' * 60)
        print(log_chunks or "(none found)")

    if not context:
        context = (
            "No relevant information found in the document; "
            "please answer based on your general medical knowledge."
        )
        
    return context

In [19]:
print("Loading vector database...")
my_faiss_index = build_or_load_vector_store(faiss_dir="./faiss_index")
    
test_question = "What is the treatment for acute myeloid leukaemia?"
print(f"\nSearching for: {test_question}")
    
retrieved_text = retrieval_with_score(
        question=test_question,
        faiss_index=my_faiss_index,
        k=3, 
        verbose=True
    )

Loading vector database...
 Loading embedding model …


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5908.88it/s]


 Embedding model ready.
 Loading existing FAISS index from './faiss_index' …
 FAISS index loaded.

Searching for: What is the treatment for acute myeloid leukaemia?

  RETRIEVED CHUNKS

[Chunk 1] Page: 161 | L2 Score: 0.2390
CR is not synonymous with cure. Standard treatment is with 4 cycles 
of intensive combination chemotherapy, each cycle followed by a 2–3-
week period of profound myelosuppression, during which good sup-
portive therapy is essential.
AML treatment consists of two phases:
1. Remission induction to ach…

[Chunk 2] Page: 164 | L2 Score: 0.2520
Acute promyelocytic leukaemia (APL/APML)
APL (FAB M3) requires a diff erent therapeutic approach that relates 
to its biology. Risk of DIC prior to and during initial therapy due to 
release  of thromboplastins from leukaemic cells is an indication for 
urgent treatment. Rapid confi rmation of prese…

[Chunk 3] Page: 651 | L2 Score: 0.2613
risk around 60–65%, and for poor risk around 15%.
• Allogeneic BMT as consolidation therapy

In [20]:
def build_prompt(
    context: str,
    history: str,
    question: str,
    template: str = "",
) -> str:
    """
    Assemble the full prompt string from context, history and question.

    A custom template can be supplied; otherwise the default clinical
    assistant template is used.
    """
    if not template:
        template = """
### System:
You are a highly knowledgeable clinical haematology assistant trained on the
Oxford Handbook of Clinical Haematology (4th Edition). Your role is to:
  • Provide accurate, concise, evidence-based answers grounded in the provided context.
  • Clearly state when information is not in the handbook and supplement with
    established medical knowledge where appropriate.
  • Use clinical terminology correctly and explain it when helpful.
  • NEVER fabricate diagnoses, drug dosages, or treatment protocols.

### Conversation history:
{history}

### Relevant context from the Oxford Handbook of Clinical Haematology:
{context}

### User question:
{question}

### Clinical Assistant:
"""
    return template.format(history=history, context=context, question=question)




In [21]:


mock_context = """CR is not synonymous with cure. Standard treatment is with 4 cycles of intensive combination chemotherapy...
Acute promyelocytic leukaemia (APL/APML) requires a different therapeutic approach..."""

mock_history = "(no history yet)"

test_question = "What is the standard treatment for AML?"

final_prompt = build_prompt(
        context=mock_context,
        history=mock_history,
        question=test_question
    )



In [22]:

print("=" * 60)
print("  FINAL PROMPT READY FOR LLM")
print("=" * 60)
print(final_prompt)

  FINAL PROMPT READY FOR LLM

### System:
You are a highly knowledgeable clinical haematology assistant trained on the
Oxford Handbook of Clinical Haematology (4th Edition). Your role is to:
  • Provide accurate, concise, evidence-based answers grounded in the provided context.
  • Clearly state when information is not in the handbook and supplement with
    established medical knowledge where appropriate.
  • Use clinical terminology correctly and explain it when helpful.
  • NEVER fabricate diagnoses, drug dosages, or treatment protocols.

### Conversation history:
(no history yet)

### Relevant context from the Oxford Handbook of Clinical Haematology:
CR is not synonymous with cure. Standard treatment is with 4 cycles of intensive combination chemotherapy...
Acute promyelocytic leukaemia (APL/APML) requires a different therapeutic approach...

### User question:
What is the standard treatment for AML?

### Clinical Assistant:



In [34]:
def llm_response(
    question: str,
    faiss_index,         
    history: list,  
    gemini_model: str = "gemini-2.5-pro", 
    history_limit: int = 5,
    verbose: bool = True
) -> str:
    """
    Full RAG turn:  retrieve → build prompt → call Gemini → return answer.
    """
    if verbose:
        print(f"\n{'=' * 60}\n  USER QUESTION\n{'=' * 60}\n{question}")

    context = retrieval_with_score(
        question=question, 
        faiss_index=faiss_index, 
        verbose=verbose
    )

    history_str = ""
    for user_msg, ai_msg in history:
        history_str += f"User: {user_msg}\nAssistant: {ai_msg}\n"
    
    if verbose:
        print(f"\n{'=' * 60}\n  CONVERSATION HISTORY\n{'=' * 60}\n{history_str or '(no history yet)'}")

    prompt = build_prompt(
        context=context,
        history=history_str,
        question=question,
    )
    
    if verbose:
        print(f"\n{'=' * 60}\n  FULL PROMPT SENT TO GEMINI\n{'=' * 60}\n{prompt}")

    answer = get_response(prompt, model_name=gemini_model)
    
    if verbose:
        print(f"\n{'=' * 60}\n  GEMINI RESPONSE\n{'=' * 60}\n{answer}")

    history.append((question, answer))
    if len(history) > history_limit:
        history.pop(0) 

    return answer


In [40]:
responce=llm_response(
    question="How is acute myeloid leukaemia treated?",
    faiss_index=my_faiss_index,
    history=[],
    gemini_model="gemini-2.5-flash",
    verbose=True
)




  USER QUESTION
How is acute myeloid leukaemia treated?

  RETRIEVED CHUNKS

[Chunk 1] Page: 161 | L2 Score: 0.2419
CR is not synonymous with cure. Standard treatment is with 4 cycles 
of intensive combination chemotherapy, each cycle followed by a 2–3-
week period of profound myelosuppression, during which good sup-
portive therapy is essential.
AML treatment consists of two phases:
1. Remission induction to ach…

[Chunk 2] Page: 164 | L2 Score: 0.2566
Acute promyelocytic leukaemia (APL/APML)
APL (FAB M3) requires a diff erent therapeutic approach that relates 
to its biology. Risk of DIC prior to and during initial therapy due to 
release  of thromboplastins from leukaemic cells is an indication for 
urgent treatment. Rapid confi rmation of prese…

[Chunk 3] Page: 651 | L2 Score: 0.2617
risk around 60–65%, and for poor risk around 15%.
• Allogeneic BMT as consolidation therapy of 1st remission is reserved 
for children in standard and poor risk groups. It is also used as a 
salvage 

In [43]:
responce

'Acute Myeloid Leukaemia (AML) treatment is intensive and consists of two main phases, with a specific approach for Acute Promyelocytic Leukaemia (APL). Treatment requires skilled supportive therapy and is confined to specialist units.\n\n**I. General AML Treatment**\n\nAML treatment aims to achieve a Complete Remission (CR), though CR is not synonymous with cure. Each cycle of intensive chemotherapy is followed by a 2–3-week period of profound myelosuppression, during which good supportive therapy is essential.\n\n1.  **Remission Induction:**\n    *   **Goal:** To achieve CR.\n    *   **Method:** Usually two courses of anthracycline-containing combination chemotherapy.\n    *   **Regimen Example (DA):**\n        *   **Daunorubicin:** 35–60 mg/m² IV daily for 3 days.\n        *   **Cytarabine:** 100 mg/m²/day as a 12-hour IV infusion or divided bolus dose twice daily for 8–10 days.\n    *   **Assessment:** Bone marrow response is assessed after 3–4 weeks.\n\n2.  **Consolidation Therapy

In [24]:
from typing import Callable

In [ ]:
def eval_pipeline(
    questions_path: str,
    answer_func: Callable[[str], str], 
    embeddings_model,                  
    threshold: float = 0.5
) -> float:

    df = pd.read_csv(questions_path)
    QA: list[tuple[np.ndarray, np.ndarray]] = []
    total_score = 0

    print(f"\n Evaluating {len(df)} questions …")
    for q in df["questions"]:
        answer = answer_func(q)
        
        w_q = np.array(embeddings_model.embed_query(q))
        w_a = np.array(embeddings_model.embed_query(answer))
        
        QA.append((w_q, w_a))

    for w_q, w_a in QA:
        cos_score = cosine_similarity(
            w_q.reshape(1, -1), w_a.reshape(1, -1)
        )[0][0]
        
        if cos_score > threshold:
            total_score += 1

    pct = (total_score / len(QA)) * 100
    print(f"  Evaluation complete: {pct:.1f}% of answers passed the threshold.")
    return pct

In [33]:
my_test_questions = [
        "What is the standard treatment for AML?",
        "How do you diagnose acute promyelocytic leukaemia?",
        "What are the symptoms of anemia?"
    ]
for test_question in my_test_questions:
    print(f"\nTesting question: {test_question}")
    retrieved_text = retrieval_with_score(
        question=test_question,
        faiss_index=my_faiss_index,
        k=3, 
        verbose=True
    )
    final_prompt = build_prompt(
        context=retrieved_text,
        history="(no history yet)",
        question=test_question
        )


Testing question: What is the standard treatment for AML?

  RETRIEVED CHUNKS

[Chunk 1] Page: 161 | L2 Score: 0.1853
CR is not synonymous with cure. Standard treatment is with 4 cycles 
of intensive combination chemotherapy, each cycle followed by a 2–3-
week period of profound myelosuppression, during which good sup-
portive therapy is essential.
AML treatment consists of two phases:
1. Remission induction to ach…

[Chunk 2] Page: 651 | L2 Score: 0.2283
risk around 60–65%, and for poor risk around 15%.
• Allogeneic BMT as consolidation therapy of 1st remission is reserved 
for children in standard and poor risk groups. It is also used as a 
salvage strategy for good risk patients who relapse.
• The role of autologous stem cell rescue following myel…

[Chunk 3] Page: 620 | L2 Score: 0.2678
spontaneous remission even after years of transfusion dependency.
Natural history
Spontaneous remission in 10–20% (even after several years). Median 
survival estimated at 30–50 years, though data 

In [54]:
from RAG_Pipeline import Pipeline

print("Initializing RAG System")
my_pipeline = Pipeline(
    gemini_model="gemini-2.5-flash",
    verbose=True
)
print("\nAsking question...")
answer = my_pipeline.llm_response(
    question="What is the standard treatment for AML?"
)

print("\nFinal Answer: ")
print(answer)

Initializing RAG System
 Loading PDF 
Loaded 871 pages.

Asking question...

  USER QUESTION
What is the standard treatment for AML?
 Loading embedding model …


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5492.45it/s]


 Embedding model ready.
 Loading existing FAISS index from './faiss_index' …
 FAISS index loaded.

  RETRIEVED CHUNKS

[Chunk 1] Page: 161 | L2 Score: 0.1853
CR is not synonymous with cure. Standard treatment is with 4 cycles 
of intensive combination chemotherapy, each cycle followed by a 2–3-
week period of profound myelosuppression, during which good sup-
portive therapy is essential.
AML treatment consists of two phases:
1. Remission induction to ach…

[Chunk 2] Page: 651 | L2 Score: 0.2283
risk around 60–65%, and for poor risk around 15%.
• Allogeneic BMT as consolidation therapy of 1st remission is reserved 
for children in standard and poor risk groups. It is also used as a 
salvage strategy for good risk patients who relapse.
• The role of autologous stem cell rescue following myel…

[Chunk 3] Page: 620 | L2 Score: 0.2678
spontaneous remission even after years of transfusion dependency.
Natural history
Spontaneous remission in 10–20% (even after several years). Median 
survival

In [55]:
answer = my_pipeline.llm_response(
    question="Hi"
)

print("\nFinal Answer: ")
print(answer)



  USER QUESTION
Hi

  RETRIEVED CHUNKS

[Chunk 1] Page: 683 | L2 Score: 0.4295
Clinical features
• HIT causes a fall in the platelet count 78d (4–14d) after a patient’s 1st 
exposure to heparin but may occur within 1–3d in a patient who has 
recently had prior exposure to heparin.
• Platelet count generally falls to 760 × 109/L but may fall to <20 × 
109/L.
• Venous and arteri…

[Chunk 2] Page: 541 | L2 Score: 0.4297
512
CHAPTER 10 Haemostasis and thrombosis
Heparin induced thrombocytopenia/
with thrombosis (HIT/T)
HIT (E see Heparin-induced thrombocytopenia (HIT), p.542) with or 
without thrombosis (HIT/T) is due to an autoantibody against heparin 
complexed with PF4, causing platelet activation. It occurs most…

[Chunk 3] Page: 571 | L2 Score: 0.4354
• Greater with UFH than with LMWH.
• All heparins used in the UK are of porcine origin.
• Frequency of HIT is greater in surgical than in medical patients.
• In orthopaedic patients given SC prophylactic heparin, the incidence 
is appro

In [56]:
answer = my_pipeline.llm_response(
    question="My name is Mazen"
)

print("\nFinal Answer: ")
print(answer)




  USER QUESTION
My name is Mazen

  RETRIEVED CHUNKS

[Chunk 1] Page: 100 | L2 Score: 0.4415
71
THALASSAEMIAS

[Chunk 2] Page: 645 | L2 Score: 0.4473
blasts on Romanowsky stains with prominent vacuoles (FAB L3 fea-
tures). Associated with chromosomal translocation involving MYC locus 
on chromosome 8 and Ig heavy chain gene on 14 (or less commonly 
with a κ or γ light chain gene on 2 or 22) with resultant dysregulation 
of MYC gene transcription;…

[Chunk 3] Page: 538 | L2 Score: 0.4584
509
MASSIVE BLOOD LOSS

[Chunk 4] Page: 13 | L2 Score: 0.4594
This page intentionally left blank


  CONVERSATION HISTORY
User: What is the standard treatment for AML?
Assistant: The standard treatment for Acute Myeloid Leukemia (AML) consists of two main phases:

1.  **Remission Induction:**
    *   **Aim:** To achieve a complete remission (CR).
    *   **Regimen:** Usually involves 2 courses of anthracycline-containing combination chemotherapy. A common regimen is "DA," which includes:
        *   **

In [57]:
answer = my_pipeline.llm_response(
    question="ايه اعراض الانيميا"
)

print("\nFinal Answer: ")
print(answer)




  USER QUESTION
ايه اعراض الانيميا

  RETRIEVED CHUNKS

[Chunk 1] Page: 66 | L2 Score: 0.3626
37
ANAEMIA IN ENDOCRINE DISEASE

[Chunk 2] Page: 147 | L2 Score: 0.3724
haematopoietic cancers, angioimmunoblastic lymphadenopathy.
• MDS.
• Collagen vascular disease, e.g. rheumatoid, SLE, GvHD.
• Infections, e.g. HIV.
• Chemotherapy.
• Surgery.
• Burns.
• Liver failure.
• Renal failure (acute and chronic).
• Anorexia nervosa.
• Fe defi ciency (uncommon).
• Aplastic an…

[Chunk 3] Page: 191 | L2 Score: 0.3756
• Progressive marrow failure: development of or worsening anaemia 
and/or thrombocytopenia.
• Autoimmune haemolytic anaemia and/or thrombocytopenia 
unresponsive to steroids.
• Massive (>6cm below LCM) or progressive splenomegaly.
• Massive (>10cm nodes or clusters) or progressive lymphadenopathy.
•…

[Chunk 4] Page: 600 | L2 Score: 0.3889
571
ANAEMIA OF PREMATURITY


  CONVERSATION HISTORY
User: What is the standard treatment for AML?
Assistant: The standard treatment for Acute Myeloid